# nanochat d8 chat SFT

This notebook is designed to run the **SFT-only** part of the `nanochat` pipeline on **Kaggle**.
The d8 base-model pretraining stage should already be finished and attached as a Kaggle Dataset.

## Before you run it

Make sure the notebook is configured with:

- **GPU enabled** (`2x Tesla T4` expected)
- **Internet enabled**
- Your uploaded Kaggle dataset **`nanochat-codes`** attached
- Your saved pretraining cache dataset **`nanochat-d8-pretrain-cache`** attached

The notebook auto-detects datasets from the mounted Kaggle path pattern
`/kaggle/input/datasets/<your-kaggle-username>/<dataset-name>`.

The `nanochat-codes` dataset is expected to contain the same code that lives under
`kaggle_dataset/nanochat/`. The saved pretraining cache is expected to contain
`base_checkpoints/d8_kaggle` and the matching `tokenizer` directory.

## What this notebook does

This notebook runs the SFT path only:

1. Kaggle environment setup and dependency installation
2. Download identity conversations for chat SFT
3. Import the saved `base_checkpoints/d8_kaggle` and matching tokenizer
4. Run supervised fine-tuning on top of the saved d8 base model
5. Evaluate and serve the SFT checkpoint


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

DATASETS_ROOT = Path('/kaggle/input/datasets')
dataset_candidates = sorted(DATASETS_ROOT.glob('*/nanochat-codes'))

# On Kaggle, this notebook expects an attached dataset named 'nanochat-codes'.
# In this repo, the contents of that mounted dataset folder correspond to the
# local folder 'kaggle_dataset/nanochat/'. The outer 'kaggle_dataset/'
# directory is only the packaging wrapper used with the Kaggle CLI.
assert dataset_candidates, (
    "Could not find an attached Kaggle dataset matching "
    "'/kaggle/input/datasets/<username>/nanochat-codes'"
)
assert len(dataset_candidates) == 1, (
    f"Expected exactly one attached 'nanochat-codes' dataset, found: {dataset_candidates}"
)
CODE_INPUT = dataset_candidates[0]

assert CODE_INPUT.exists(), f'Code dataset not found: {CODE_INPUT}'
print('Code input:', CODE_INPUT)
print('Top-level code files:', sorted(p.name for p in CODE_INPUT.iterdir()))


In [ ]:
WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['NANOCHAT_DTYPE'] = 'float16'
os.environ['PYTHONFAULTHANDLER'] = '1'


print('Working repo:', WORK_REPO)
print('Nanochat base dir:', WORK_CACHE)
print('HF cache:', HF_CACHE)
print('Repo contents:', sorted(p.name for p in WORK_REPO.iterdir()))
print('NANOCHAT_DTYPE:', os.environ['NANOCHAT_DTYPE'])


In [ ]:
# Input HuggingFace token
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

In [ ]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'python-dotenv>=1.2.1',
    'pyyaml>=6.0.0',
    'regex>=2025.9.1',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'scipy>=1.15.3',
    'tabulate>=0.9.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)


In [ ]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


## 1. Download Identity Conversations for Chat SFT

This small JSONL file gives the model a lightweight assistant identity before chat fine-tuning.


In [ ]:
cmd = [
    'curl',
    '-L',
    '-o', str(WORK_CACHE / 'identity_conversations.jsonl'),
    'https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)
print('Saved:', WORK_CACHE / 'identity_conversations.jsonl')


## 2. Import Saved d8 Base Checkpoints

This SFT notebook does not run pretraining. It imports the already-saved d8 base checkpoint from the attached Kaggle Dataset and places it under `WORK_CACHE/base_checkpoints`, where `scripts.chat_sft_updated` expects to find `base:d8_kaggle`.

The matching tokenizer is copied from the same saved cache because model loading checks tokenizer compatibility.


In [ ]:
BASE_CACHE_DATASET = 'nanochat-d8-pretrain-cache'
MODEL_TAG = 'd8_kaggle'

cache_candidates = sorted(DATASETS_ROOT.glob(f'*/{BASE_CACHE_DATASET}'))
assert cache_candidates, (
    'Could not find an attached Kaggle dataset matching '
    f"'/kaggle/input/datasets/<username>/{BASE_CACHE_DATASET}'"
)
assert len(cache_candidates) == 1, (
    f"Expected exactly one attached '{BASE_CACHE_DATASET}' dataset, found: {cache_candidates}"
)
BASE_CACHE_INPUT = cache_candidates[0]

required_cache_dirs = ['base_checkpoints', 'tokenizer']
for dirname in required_cache_dirs:
    source = BASE_CACHE_INPUT / dirname
    destination = WORK_CACHE / dirname
    assert source.exists(), f'Missing {dirname} in saved pretraining cache: {source}'
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(source, destination)

base_dir = WORK_CACHE / 'base_checkpoints' / MODEL_TAG
assert base_dir.exists(), f'Missing imported base checkpoint directory: {base_dir}'
model_files = sorted(base_dir.glob('model_*.pt'))
meta_files = sorted(base_dir.glob('meta_*.json'))
assert model_files, f'No model_*.pt files found in {base_dir}'
assert meta_files, f'No meta_*.json files found in {base_dir}'

print('Imported saved cache:', BASE_CACHE_INPUT)
print('Imported base checkpoint:', base_dir)
print('Latest model file:', model_files[-1].name)
print('Tokenizer files:', sorted(p.name for p in (WORK_CACHE / 'tokenizer').iterdir()))


## 3. Chat SFT on Top of the Saved d8 Base Model

Use the Kaggle-safe SFT entrypoint here:

- run `scripts.chat_sft_updated`
- load `base:d8_kaggle` from the imported `base_checkpoints`
- use `torchrun --nproc_per_node=2`
- keep `--run=dummy` to avoid wandb login requirements
- use `--device-batch-size=8`
- set `--total-batch-size=16384`
- keep `--mmlu-epochs=1` and `--gsm8k-epochs=1` for a lighter first pass
- disable periodic eval and ChatCORE for this run


In [ ]:
# Real d8 SFT on Kaggle 2xT4
# Starts from base:d8_kaggle and saves to chatsft_checkpoints/d8_kaggle
SFT_STEPS = 5000

# ---------------------------------------------------------------------
# Verify base checkpoint is finite before SFT.

base_dir = WORK_CACHE / "base_checkpoints" / "d8_kaggle"
latest_model = sorted(base_dir.glob("model_*.pt"))[-1]
state = torch.load(latest_model, map_location="cpu")
bad = [
    name for name, tensor in state.items()
    if torch.is_floating_point(tensor) and not torch.isfinite(tensor).all()
]
assert not bad, f"Base checkpoint has non-finite tensors: {bad[:8]}"
del state
print("Base checkpoint finite:", latest_model)

# ---------------------------------------------------------------------
# Runtime env

env = os.environ.copy()
env["NANOCHAT_DTYPE"] = "float16"
env["NANOCHAT_DISABLE_COMPILE"] = "1"
env["TORCH_COMPILE_DISABLE"] = "1"

env["NANOCHAT_ADAMW_ALLREDUCE"] = "1"
env["NANOCHAT_SERIAL_OPTIM_COMM"] = "1"

env["OMP_NUM_THREADS"] = "1"
env["PYTHONUNBUFFERED"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
env["TORCH_NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["TORCH_FR_BUFFER_SIZE"] = "1048576"
env["NCCL_DEBUG"] = "WARN"

env["NANOCHAT_DEBUG_DIST"] = "0"
env["NANOCHAT_DEBUG_STDOUT"] = "0"
env["NANOCHAT_DEBUG_STACK"] = "0"

# ---------------------------------------------------------------------
# Run SFT

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "-m", "scripts.chat_sft_updated",
    "--",

    "--model-tag=d8_kaggle",
    "--run=dummy",

    "--max-seq-len=1024",
    "--device-batch-size=8",
    "--total-batch-size=16384",

    f"--num-iterations={SFT_STEPS}",
    "--load-optimizer=0",

    "--embedding-lr=0.03",
    "--unembedding-lr=0.0008",
    "--matrix-lr=0.002",
    "--init-lr-frac=0.2",
    "--warmup-ratio=0.05",
    "--warmdown-ratio=0.2",

    "--mmlu-epochs=1",
    "--gsm8k-epochs=1",

    "--eval-every=-1",
    "--eval-tokens=524288",
    "--chatcore-every=-1",
]

print("Running command:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=WORK_REPO,
    env=env,
    check=True,
)


## 4. Evaluate the SFT Chat Model

Use a reduced 2-GPU chat evaluation first:

- source is `sft`
- load model tag `d8_kaggle`
- limit to `--max-problems=50`
- use `--batch-size=2` for categorical tasks


In [ ]:
env = os.environ.copy()
env["NANOCHAT_DTYPE"] = "float16"
env["NANOCHAT_DISABLE_COMPILE"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "-m", "scripts.chat_eval",
    "--",
    "-i", "sft",
    "-g", "d8_kaggle",
    "-x", "50",
    "-b", "2",
]

print("Running command:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=WORK_REPO,
    env=env,
    check=True,
)

In [ ]:
# Optional: save the SFT checkpoint cache as a Kaggle Dataset.
import json

SFT_MODEL_TAG = 'd8_kaggle'
SFT_SOURCE_DIR = WORK_CACHE / 'chatsft_checkpoints' / SFT_MODEL_TAG
TOKENIZER_SOURCE_DIR = WORK_CACHE / 'tokenizer'
SFT_CACHE_DIR = Path('/kaggle/working/nanochat_d8_sft_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-sft-cache'
TITLE = 'nanochat d8 sft cache'
VERSION_MESSAGE = f'Save {SFT_MODEL_TAG} chat SFT checkpoint cache'

assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
assert SFT_SOURCE_DIR.exists(), f'Missing SFT checkpoint directory: {SFT_SOURCE_DIR}'
assert TOKENIZER_SOURCE_DIR.exists(), f'Missing tokenizer directory: {TOKENIZER_SOURCE_DIR}'
assert sorted(SFT_SOURCE_DIR.glob('model_*.pt')), f'No model_*.pt files found in {SFT_SOURCE_DIR}'
assert sorted(SFT_SOURCE_DIR.glob('meta_*.json')), f'No meta_*.json files found in {SFT_SOURCE_DIR}'

if SFT_CACHE_DIR.exists():
    shutil.rmtree(SFT_CACHE_DIR)
SFT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(WORK_CACHE / 'chatsft_checkpoints', SFT_CACHE_DIR / 'chatsft_checkpoints')
shutil.copytree(TOKENIZER_SOURCE_DIR, SFT_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = SFT_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(SFT_CACHE_DIR)], check=False)

cmd = [
    'kaggle', 'datasets', 'version',
    '-p', str(SFT_CACHE_DIR),
    '-m', VERSION_MESSAGE,
    '--dir-mode', 'tar',
]

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload/version failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


## 5. Serve the SFT Chat Model

Run this after `chatsft_checkpoints/d8_kaggle` exists. The first cell starts or restarts the local NanoChat web server from the SFT checkpoint. The second cell provides a small notebook chat loop against that server.


In [ ]:
import time
import requests

SFT_CHECKPOINT_DIR = WORK_CACHE / "chatsft_checkpoints"
SFT_MODEL_TAG = "d8_kaggle"
SERVER_PORT = 8000
BASE_URL = f"http://127.0.0.1:{SERVER_PORT}"

model_dir = SFT_CHECKPOINT_DIR / SFT_MODEL_TAG
assert model_dir.exists(), f"Missing SFT checkpoint directory: {model_dir}"
assert sorted(model_dir.glob("model_*.pt")), f"No model_*.pt files found in {model_dir}"
assert sorted(model_dir.glob("meta_*.json")), f"No meta_*.json files found in {model_dir}"

try:
    if server.poll() is None:
        print("Stopping existing NanoChat server...")
        server.terminate()
        server.wait(timeout=10)
        print("Stopped existing server.")
except NameError:
    pass
except Exception as exc:
    print("Could not stop existing server cleanly:", exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print("Killed existing server.")
    except Exception:
        pass

messages = []

server_env = os.environ.copy()
server_env["NANOCHAT_DTYPE"] = "float16"
server_env["NANOCHAT_DISABLE_COMPILE"] = "1"
server_env["TORCH_COMPILE_DISABLE"] = "1"
server_env["OMP_NUM_THREADS"] = "1"

cmd = [
    sys.executable,
    "-m", "scripts.chat_web",
    "--checkpoint-dir", str(SFT_CHECKPOINT_DIR),
    "--model-tag", SFT_MODEL_TAG,
    "--num-gpus", "1",
    "--port", str(SERVER_PORT),
]

print("Starting NanoChat server:")
print(" ".join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f"Server process started with PID {server.pid}.")

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f"NanoChat server exited early with code {server.returncode}")
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f"NanoChat server is ready: {BASE_URL}")
else:
    print(f"NanoChat server is still loading. Wait a bit, then use: {BASE_URL}")


In [ ]:
import json
import requests

BASE = globals().get("BASE_URL", "http://127.0.0.1:8000")
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "temperature": temperature,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    print("Assistant:", end=" ", flush=True)
    answer = ""

    with requests.post(f"{BASE}/chat/completions", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue

            data = json.loads(line[len("data: "):])
            if data.get("done"):
                break

            token = data.get("token", "")
            answer += token
            print(token, end="", flush=True)

    print()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(f"Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.")
while True:
    prompt = input("\nYou: ")
    command = prompt.strip().lower()
    if command in {"q", "quit", "exit"}:
        break
    if command == "reset":
        messages.clear()
        print("Chat history cleared.")
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

# Fallback for cases where the notebook variable was lost but chat_web is still running.
try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')
